# local_backtest 本地研究入口

这个 Notebook 用于本地交互式观察，不替代正式报告图表。正式 PNG 图表仍由 `engine.charts.generate_report_charts("reports")` 生成。

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

## 净值与回撤

读取 `reports/equity_curve.csv`。如果文件不存在，只提示下一步动作，不生成替代数据。

In [ ]:
equity_path = Path("reports/equity_curve.csv")

if not equity_path.exists():
    display(Markdown("未找到 `reports/equity_curve.csv`。请先运行一次本地回测并输出净值曲线 CSV；本 Notebook 不生成替代数据。"))
else:
    equity = pd.read_csv(equity_path)
    if "trade_date" not in equity.columns:
        display(Markdown("`reports/equity_curve.csv` 缺少 `trade_date` 字段，无法绘制时间序列。"))
    else:
        equity = equity.copy()
        equity["trade_date"] = pd.to_datetime(equity["trade_date"], errors="coerce")
        equity = equity.dropna(subset=["trade_date"]).sort_values("trade_date")

        if "nav" in equity.columns:
            equity["nav"] = pd.to_numeric(equity["nav"], errors="coerce")
        elif "total_value" in equity.columns:
            total_value = pd.to_numeric(equity["total_value"], errors="coerce")
            first_value = total_value.dropna().iloc[0] if not total_value.dropna().empty else None
            if first_value and first_value != 0:
                equity["nav"] = total_value / first_value

        if "nav" not in equity.columns:
            display(Markdown("`reports/equity_curve.csv` 需要 `nav` 或 `total_value` 字段。"))
        else:
            if "drawdown" in equity.columns:
                equity["drawdown"] = pd.to_numeric(equity["drawdown"], errors="coerce")
            else:
                equity["drawdown"] = equity["nav"] / equity["nav"].cummax() - 1.0

            fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
            axes[0].plot(equity["trade_date"], equity["nav"], color="#1f77b4", linewidth=1.6)
            axes[0].set_title("Equity Curve")
            axes[0].set_ylabel("NAV")
            axes[1].fill_between(equity["trade_date"], equity["drawdown"] * 100, 0, color="#c0392b", alpha=0.28)
            axes[1].set_title("Drawdown")
            axes[1].set_ylabel("Drawdown (%)")
            axes[1].set_xlabel("Date")
            fig.autofmt_xdate()
            fig.tight_layout()
            plt.show()

## 可选 K 线示例

如需查看 K 线，把用户自备 OHLCV CSV 放到 `data/ohlcv.csv`。示例只在字段完整时绘制，不用净值曲线伪造 K 线。

In [ ]:
ohlcv_path = Path("data/ohlcv.csv")
required_lower = {"open", "high", "low", "close", "volume"}
required_title = {"Open", "High", "Low", "Close", "Volume"}

if not ohlcv_path.exists():
    display(Markdown("未找到 `data/ohlcv.csv`，跳过 K 线示例。请使用自备 OHLCV 数据。"))
else:
    raw_ohlcv = pd.read_csv(ohlcv_path)
    columns = set(raw_ohlcv.columns)
    if required_lower.issubset(columns):
        rename_map = {"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"}
    elif required_title.issubset(columns):
        rename_map = {}
    else:
        display(Markdown("`data/ohlcv.csv` 缺少完整 OHLCV 字段，跳过 K 线示例。"))
        rename_map = None

    if rename_map is not None:
        date_column = "trade_date" if "trade_date" in raw_ohlcv.columns else "date" if "date" in raw_ohlcv.columns else "Date" if "Date" in raw_ohlcv.columns else None
        if date_column is None:
            display(Markdown("`data/ohlcv.csv` 缺少 `trade_date`、`date` 或 `Date` 日期字段，跳过 K 线示例。"))
        else:
            import mplfinance as mpf

            ohlcv = raw_ohlcv.rename(columns=rename_map).copy()
            ohlcv[date_column] = pd.to_datetime(ohlcv[date_column], errors="coerce")
            ohlcv = ohlcv.dropna(subset=[date_column]).set_index(date_column).sort_index()
            ohlcv = ohlcv[["Open", "High", "Low", "Close", "Volume"]].apply(pd.to_numeric, errors="coerce").dropna()

            if ohlcv.empty:
                display(Markdown("`data/ohlcv.csv` 没有有效 OHLCV 行，跳过 K 线示例。"))
            else:
                mpf.plot(ohlcv, type="candle", volume=True, style="yahoo", title="OHLCV Candles")